# ADMET-AI CPU 예제
대규모 화합물 라이브러리의 ADMET 물성을 일괄 예측한다.
## 실습 개요
- **활용 분야** — ADMET 예측
- **카테고리** — Screening · Property
- **모델 유형** — Chemprop (GNN)
- **발표 연도** — 2024
- **라이선스** — MIT
**관련 논문**

- [ADMET-AI: a machine learning ADMET platform for evaluation of large-scale chemical libraries](https://academic.oup.com/bioinformatics/article/40/7/btae416/7698030) — Kyle Swanson 외, Bioinformatics, 2024

> **CPU 예제입니다.** 아래 예제는 한 개의 SMILES 또는 짧은 펩타이드 입력을 사용합니다. 플랫폼과 입력 방식·패키지 버전·산출물 종류가 다를 수 있습니다.

**노트북 패키지 버전:** 2.0.1. **플랫폼 카탈로그 버전:** 1.3.1. 서로 다른 버전의 수치를 동일한 결과로 비교하지 마세요. 설치된 버전은 결과 파일에 기록됩니다.

## 실행 조건

- **코드 저장소** — https://github.com/swansonk14/admet_ai
- **라이브러리** — 아래 환경 안내 참고
- **플랫폼에서 바로 실행** — https://lab.curieus.net/Analysis/admet-ai

## 실행 제한사항 및 유의사항
- CPU 만으로 동작한다. GPU 런타임이 필요 없다.
- 분자 수가 많으면 시간이 늘어난다. 수천 건 단위는 나눠서 실행할 것.
- 세션이 끊기면 설치한 패키지와 내려받은 가중치가 모두 사라진다. 재접속하면 설치부터 다시 해야 한다.
- 무료 런타임은 유휴 상태가 이어지면 연결이 끊긴다. 긴 실행은 중간 결과를 Google Drive 에 저장하는 편이 안전하다.

## 실행 절차

CPU 런타임에서 환경 준비 → 입력값 설정 → 실행 → 결과 확인 순서로 실행하세요. GPU 설정은 필요하지 않습니다.

이 예제는 ADMET-AI v2로 아스피린 한 분자의 물성을 예측합니다. 논문 및 플랫폼의 v1 계열 결과를 재현하는 예제가 아닙니다.

## 환경 준비

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path
from sysconfig import get_path
SCRIPTS = Path(get_path("scripts"))
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # CPU example
os.environ['MPLBACKEND'] = 'Agg'  # CLI runs outside the notebook kernel
print('Python:', platform.python_version())
subprocess.run([sys.executable, "-m", "pip", "install", "admet-ai==2.0.1"], check=True)

## 입력값 설정

In [ ]:
SMILES = "CC(=O)Oc1ccccc1C(=O)O"  # @param {type:"string"}
from pathlib import Path
import pandas as pd
from rdkit import Chem
if Chem.MolFromSmiles(SMILES) is None:
    raise ValueError("Invalid SMILES")
pd.DataFrame({"smiles": [SMILES]}).to_csv("input.csv", index=False)

## 실행

In [ ]:
from pathlib import Path
Path("preds.csv").unlink(missing_ok=True)  # never display an old result after a failed run
subprocess.run([str(SCRIPTS / "admet_predict"), "--data_path", "input.csv", "--save_path", "preds.csv", "--smiles_column", "smiles"], check=True, timeout=600)

## 결과 확인

In [ ]:
import json
import pandas as pd
from importlib.metadata import version
from IPython.display import display
result_path = Path("preds.csv")
if not result_path.is_file() or result_path.stat().st_size == 0:
    raise RuntimeError("Prediction output was not created")
predictions = pd.read_csv(result_path)
expected_rows = 1
if len(predictions) != expected_rows or predictions.select_dtypes(include="number").dropna(how="all").empty:
    raise RuntimeError("Prediction output is empty or incomplete")
required_scores = ["AMES","hERG"]
import numpy as np
if not set(required_scores).issubset(predictions.columns):
    raise RuntimeError("Expected model prediction columns are missing")
scores = predictions[required_scores].apply(pd.to_numeric, errors="raise")
if not np.isfinite(scores.to_numpy()).all():
    raise RuntimeError("Model prediction scores contain missing or non-finite values")
display(predictions)
environment = {"python": platform.python_version(), "package": "admet-ai", "version": version("admet-ai"), "platform_catalog_version": "1.3.1"}
Path("environment.json").write_text(json.dumps(environment, indent=2))
print(environment)
print("Saved: preds.csv, environment.json")

## 결과 해석

이 예제의 산출물은 물성별 예측 CSV입니다. 확률·회귀 값은 열마다 의미와 단위가 다르므로 저장소 문서를 확인하세요. 실험 측정값이나 의약품 안전성 판정이 아닙니다.

원본 논문과 저장소 문서의 지표 정의를 함께 확인하세요.